In [2]:
# pip install selenium webdriver-manager pandas openpyxl

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time

URL = "https://www.directdb.co.kr/comm/cst/reptnqtn/repetitionQuestionView.do"

def clean(text):
    return " ".join((text or "").split())

def get_driver():
    options = webdriver.ChromeOptions()
    options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--blink-settings=imagesEnabled=false")
    options.page_load_strategy = "eager"

    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

def extract_current_page(driver):
    return driver.execute_script("""
        function clean(t){
            return (t || '').replace(/\\s+/g, ' ').trim();
        }

        const out = [];
        document.querySelectorAll('#faqList li.base-acc-item').forEach(li => {
            const menu = clean(li.querySelector('span.menu')?.innerText);
            const fullQ = clean(li.querySelector('strong.faq-tit')?.innerText);
            const answer =
                clean(li.querySelector('p.faq-txt')?.innerText) ||
                clean(li.querySelector('div.base-acc-cont')?.innerText);

            let question = fullQ;
            if (menu && fullQ.startsWith(menu)) {
                question = clean(fullQ.slice(menu.length));
            }

            if (menu && question && answer) {
                out.push({
                    category: menu,
                    question: question,
                    answer: answer
                });
            }
        });
        return out;
    """)

def get_first_question(driver):
    rows = extract_current_page(driver)
    if not rows:
        return ""
    return rows[0]["question"]

def click_page_number(driver, page_num):
    return driver.execute_script("""
        const target = arguments[0].toString();

        function txt(el){
            return (el.innerText || '').replace(/\\s+/g,' ').trim();
        }

        const candidates = [...document.querySelectorAll('a, button, span')];

        // 페이지 번호만 가진 요소 우선 탐색
        let el = candidates.find(e => txt(e) === target);

        // 못 찾으면 부모/형제 포함해서 재탐색
        if (!el) {
            el = [...document.querySelectorAll('*')].find(e => txt(e) === target);
        }

        if (el) {
            el.click();
            return true;
        }
        return false;
    """, page_num)

def wait_page_changed(driver, old_first_question, timeout=6):
    end = time.time() + timeout
    while time.time() < end:
        new_first_question = get_first_question(driver)
        if new_first_question and new_first_question != old_first_question:
            return True
        time.sleep(0.3)
    return False

def crawl_db_all_pages():
    driver = get_driver()
    wait = WebDriverWait(driver, 10)

    driver.get(URL)
    wait.until(lambda d: len(d.find_elements(By.CSS_SELECTOR, "#faqList li.base-acc-item")) > 0)

    all_rows = []
    seen = set()

    # 1페이지 먼저 수집
    rows = extract_current_page(driver)
    for row in rows:
        key = (row["category"], row["question"], row["answer"])
        if key not in seen:
            seen.add(key)
            all_rows.append(row)
    print(f"[페이지 1] {len(rows)}건 / 누적 {len(all_rows)}건")

    # 2~20페이지 순회
    for page_num in range(2, 21):
        old_first = get_first_question(driver)

        clicked = click_page_number(driver, page_num)
        if not clicked:
            print(f"[페이지 {page_num}] 클릭 실패")
            continue

        changed = wait_page_changed(driver, old_first, timeout=6)
        if not changed:
            print(f"[페이지 {page_num}] 페이지 변경 감지 실패")
            continue

        rows = extract_current_page(driver)
        added = 0
        for row in rows:
            key = (row["category"], row["question"], row["answer"])
            if key not in seen:
                seen.add(key)
                all_rows.append(row)
                added += 1

        print(f"[페이지 {page_num}] {added}건 추가 / 누적 {len(all_rows)}건")

    driver.quit()

    for i, row in enumerate(all_rows, 1):
        print(f"\\n[{i}]")
        print("카테고리:", row["category"])
        print("질문:", row["question"])
        print("답변:", row["answer"])
        print("-" * 100)

    print(f"\\n총 {len(all_rows)}건 추출 완료")

    df = pd.DataFrame(all_rows, columns=["category", "question", "answer"])
    df.to_excel("db_direct_faq_all_pages.xlsx", index=False)
    print("엑셀 저장 완료: db_direct_faq_all_pages.xlsx")

if __name__ == "__main__":
    crawl_db_all_pages()

[페이지 1] 10건 / 누적 10건
[페이지 2] 10건 추가 / 누적 20건
[페이지 3] 10건 추가 / 누적 30건
[페이지 4] 10건 추가 / 누적 40건
[페이지 5] 10건 추가 / 누적 50건
[페이지 6] 10건 추가 / 누적 60건
[페이지 7] 10건 추가 / 누적 70건
[페이지 8] 10건 추가 / 누적 80건
[페이지 9] 10건 추가 / 누적 90건
[페이지 10] 10건 추가 / 누적 100건
[페이지 11] 클릭 실패
[페이지 12] 클릭 실패
[페이지 13] 클릭 실패
[페이지 14] 클릭 실패
[페이지 15] 클릭 실패
[페이지 16] 클릭 실패
[페이지 17] 클릭 실패
[페이지 18] 클릭 실패
[페이지 19] 클릭 실패
[페이지 20] 클릭 실패
\n[1]
카테고리: 상품안내
질문: 1035건강보험은 어떤 상품인가요?
답변: 1035건강보험은 10~35세 대상으로만 가입 가능한 종합 건강보험입니다. 3대질환(암, 뇌, 심장)을 비롯하여 수술비, 입원비, 상해 등 하나의 상품으로 다양한 위험을 보장하는 상품입니다.(특약 가입 시) 상품 주요특징으로는 가입유형이 ‘일반고지’와 건강상태에 따라 보험료 부담을 덜어주는 ‘건강고지‘로 구성되어 있습니다.
----------------------------------------------------------------------------------------------------
\n[2]
카테고리: 상품안내
질문: 건강고지는 일반고지와 어떻게 다른가요?
답변: 1035건강보험은 10~35세 대상으로만 가입 가능한 종합 건강보험입니다. 3대질환(암, 뇌, 심장)을 비롯하여 수술비, 입원비, 상해 등 하나의 상품으로 다양한 위험을 보장하는 상품입니다.(특약 가입 시) 상품 주요특징으로는 가입유형이 ‘일반고지’와 건강상태에 따라 보험료 부담을 덜어주는 ‘건강고지‘로 구성되어 있습니다.
------------------------------------------------------